# 12 — Insights and Reports


In [1]:
# Imports and Paths

from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import (
    getSampleStyleSheet,
    ParagraphStyle,
)
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    Table,
    TableStyle,
    Image as PDFImage,
)

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_FILE = ROOT / "artifacts" / "cleaned_data.parquet"
MODEL_DIR = ROOT / "artifacts" / "models"
UNSUPERVISED_DIR = ROOT / "artifacts" / "unsupervised"
EXPLAIN_DIR = ROOT / "artifacts" / "explainability"
CHART_DIR = ROOT / "artifacts" / "charts"
REPORT_DIR = ROOT / "artifacts" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Load projects artifacts

df = pd.read_parquet(DATA_FILE)

with open(
    MODEL_DIR / "final_selection_summary.json",
    "r",
    encoding="utf-8",
) as file:
    final_model_summary = json.load(file)

with open(
    MODEL_DIR / "calibrated_model_metrics.json",
    "r",
    encoding="utf-8",
) as file:
    calibrated_metrics = json.load(file)

with open(
    EXPLAIN_DIR / "explanation_summary.json",
    "r",
    encoding="utf-8",
) as file:
    explanation_summary = json.load(file)

with open(
    UNSUPERVISED_DIR / "unsupervised_summary.json",
    "r",
    encoding="utf-8",
) as file:
    unsupervised_summary = json.load(file)

grouped_shap_df = pd.read_csv(
    EXPLAIN_DIR / "grouped_shap_importance.csv"
)

permutation_df = pd.read_csv(
    EXPLAIN_DIR / "permutation_importance.csv"
)

cluster_profile_df = pd.read_csv(
    UNSUPERVISED_DIR / "cluster_profiles.csv"
)

robust_validation_df = pd.read_csv(
    MODEL_DIR / "robust_validation_results.csv"
)

print("All project artifacts loaded.")

All project artifacts loaded.


In [3]:
# Dataset overview

TARGET_COLUMN = "purchased"

dataset_overview = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "missing_values": int(
        df.isna().sum().sum()
    ),
    "duplicate_rows": int(
        df.duplicated().sum()
    ),
    "numeric_columns": int(
        len(
            df.select_dtypes(
                include=np.number
            ).columns
        )
    ),
    "categorical_columns": int(
        len(
            df.select_dtypes(
                exclude=np.number
            ).columns
        )
    ),
    "target_column": TARGET_COLUMN,
    "purchase_rate": float(
        df[TARGET_COLUMN].mean()
    ),
}

dataset_overview

{'rows': 500,
 'columns': 7,
 'missing_values': 0,
 'duplicate_rows': 0,
 'numeric_columns': 6,
 'categorical_columns': 1,
 'target_column': 'purchased',
 'purchase_rate': 0.608}

In [4]:
# Generate Insights 

top_shap_features = (
    grouped_shap_df.head(3)[
        "original_feature"
    ].tolist()
)

highest_purchase_cluster = (
    cluster_profile_df.loc[
        cluster_profile_df[
            "purchase_rate_percentage"
        ].idxmax()
    ]
)

lowest_purchase_cluster = (
    cluster_profile_df.loc[
        cluster_profile_df[
            "purchase_rate_percentage"
        ].idxmin()
    ]
)

business_insights = [
    (
        f"The dataset contains "
        f"{dataset_overview['rows']:,} records "
        f"and {dataset_overview['columns']} columns."
    ),
    (
        f"The overall purchase rate is "
        f"{dataset_overview['purchase_rate'] * 100:.2f}%."
    ),
    (
        f"The validation-selected model is "
        f"{final_model_summary['final_model']}."
    ),
    (
        f"After probability calibration, the model achieved "
        f"{calibrated_metrics['accuracy'] * 100:.2f}% accuracy, "
        f"{calibrated_metrics['macro_f1']:.4f} macro F1 and "
        f"{calibrated_metrics['roc_auc']:.4f} ROC-AUC."
    ),
    (
        f"Probability calibration reduced log loss to "
        f"{calibrated_metrics['log_loss']:.4f} and the "
        f"Brier score to "
        f"{calibrated_metrics['brier_score']:.4f}."
    ),
    (
        f"The most influential purchase features are "
        f"{', '.join(top_shap_features)}."
    ),
    (
        f"Customer cluster "
        f"{int(highest_purchase_cluster['kmeans_cluster'])} "
        f"has the highest observed purchase rate at "
        f"{highest_purchase_cluster['purchase_rate_percentage']:.2f}%."
    ),
    (
        f"Customer cluster "
        f"{int(lowest_purchase_cluster['kmeans_cluster'])} "
        f"has the lowest observed purchase rate at "
        f"{lowest_purchase_cluster['purchase_rate_percentage']:.2f}%."
    ),
    (
        f"K-Means produced "
        f"{unsupervised_summary['selected_k']} clusters, "
        f"but its silhouette score was only "
        f"{unsupervised_summary['kmeans_silhouette_score']:.4f}; "
        f"the segments overlap and require business validation."
    ),
    (
        f"Isolation Forest identified "
        f"{unsupervised_summary['anomaly_records']} "
        f"potential anomalies."
    ),
]

for number, insight in enumerate(
    business_insights,
    start=1,
):
    print(f"{number}. {insight}")

1. The dataset contains 500 records and 7 columns.
2. The overall purchase rate is 60.80%.
3. The validation-selected model is Decision Tree.
4. After probability calibration, the model achieved 86.00% accuracy, 0.8586 macro F1 and 0.9151 ROC-AUC.
5. Probability calibration reduced log loss to 0.3571 and the Brier score to 0.1071.
6. The most influential purchase features are visits, income, satisfaction.
7. Customer cluster 5 has the highest observed purchase rate at 100.00%.
8. Customer cluster 6 has the lowest observed purchase rate at 10.77%.
9. K-Means produced 8 clusters, but its silhouette score was only 0.1771; the segments overlap and require business validation.
10. Isolation Forest identified 25 potential anomalies.


In [5]:
# Recommendations and Limitations

recommendations = [
    (
        "Prioritize visits, income and satisfaction "
        "when designing customer-engagement strategies."
    ),
    (
        "Investigate customers with high visit activity "
        "but low purchase probability."
    ),
    (
        "Review potential anomalies before removing them; "
        "they may represent valuable customer groups."
    ),
    (
        "Use the calibrated model for probabilities and "
        "the base Decision Tree for SHAP explanations."
    ),
    (
        "Collect more real business data before production "
        "deployment or final segment naming."
    ),
]

limitations = [
    (
        "The demonstration dataset is synthetic and "
        "contains only 500 records."
    ),
    (
        "The holdout set was inspected during development, "
        "so it is not a completely untouched external audit."
    ),
    (
        "Correlation, SHAP and partial dependence describe "
        "model associations, not causal effects."
    ),
    (
        "The clustering silhouette score is low, indicating "
        "overlapping customer groups."
    ),
    (
        "Model performance may change when data distribution "
        "changes over time."
    ),
]

print("Recommendations:")

for item in recommendations:
    print("-", item)

print("\nLimitations:")

for item in limitations:
    print("-", item)

Recommendations:
- Prioritize visits, income and satisfaction when designing customer-engagement strategies.
- Investigate customers with high visit activity but low purchase probability.
- Review potential anomalies before removing them; they may represent valuable customer groups.
- Use the calibrated model for probabilities and the base Decision Tree for SHAP explanations.
- Collect more real business data before production deployment or final segment naming.

Limitations:
- The demonstration dataset is synthetic and contains only 500 records.
- The holdout set was inspected during development, so it is not a completely untouched external audit.
- Correlation, SHAP and partial dependence describe model associations, not causal effects.
- The clustering silhouette score is low, indicating overlapping customer groups.
- Model performance may change when data distribution changes over time.


In [6]:
# Natural-language Q&A engine

from src.ai.qa import answer_question


def answer_business_question(question):
    question_lower = question.lower().strip()

    if any(
        phrase in question_lower
        for phrase in [
            "best model",
            "selected model",
            "which model",
        ]
    ):
        return (
            f"The validation-selected model is "
            f"{final_model_summary['final_model']}. "
            f"The deployed version uses sigmoid "
            f"probability calibration."
        )

    if any(
        phrase in question_lower
        for phrase in [
            "important feature",
            "important features",
            "top feature",
            "top features",
            "influential feature",
            "influential features",
        ]
    ):
        return (
            "The most influential features are "
            + ", ".join(top_shap_features)
            + "."
        )

    if any(
        phrase in question_lower
        for phrase in [
            "model accuracy",
            "model performance",
            "accuracy",
        ]
    ):
        return (
            f"The calibrated model achieved "
            f"{calibrated_metrics['accuracy'] * 100:.2f}% "
            f"accuracy, macro F1 of "
            f"{calibrated_metrics['macro_f1']:.4f}, "
            f"and ROC-AUC of "
            f"{calibrated_metrics['roc_auc']:.4f}."
        )

    if any(
        phrase in question_lower
        for phrase in [
            "cluster",
            "clusters",
            "segmentation",
            "segments",
        ]
    ):
        return (
            f"K-Means selected "
            f"{unsupervised_summary['selected_k']} clusters. "
            f"The silhouette score is "
            f"{unsupervised_summary['kmeans_silhouette_score']:.4f}, "
            f"so the clusters overlap and should be "
            f"interpreted carefully."
        )

    if any(
        phrase in question_lower
        for phrase in [
            "anomaly",
            "anomalies",
            "anomalous",
            "outlier",
            "outliers",
        ]
    ):
        return (
            f"Isolation Forest identified "
            f"{unsupervised_summary['anomaly_records']} "
            f"potential anomalies, representing "
            f"{unsupervised_summary['anomaly_percentage']:.2f}% "
            f"of the dataset."
        )

    # This must remain outside every preceding if block.
    return answer_question(
        df,
        question,
    )

In [7]:
# Test Natural language Questions

sample_questions = [
    "How many rows are in the dataset?",
    "What is the average income?",
    "How many missing values are there?",
    "Which is the best model?",
    "What are the most important features?",
    "What is the model accuracy?",
    "How many clusters were created?",
    "How many anomalies were found?",
]

qa_results = []

for question in sample_questions:
    answer = answer_business_question(
        question
    )

    qa_results.append({
        "question": question,
        "answer": answer,
    })

    print("Question:", question)
    print("Answer:", answer)
    print("-" * 80)

qa_results_df = pd.DataFrame(
    qa_results
)

display(qa_results_df)

Question: How many rows are in the dataset?
Answer: The dataset has 500 rows.
--------------------------------------------------------------------------------
Question: What is the average income?
Answer: The mean of income is 85,746.1120.
--------------------------------------------------------------------------------
Question: How many missing values are there?
Answer: The dataset has no missing values.
--------------------------------------------------------------------------------
Question: Which is the best model?
Answer: The validation-selected model is Decision Tree. The deployed version uses sigmoid probability calibration.
--------------------------------------------------------------------------------
Question: What are the most important features?
Answer: The most influential features are visits, income, satisfaction.
--------------------------------------------------------------------------------
Question: What is the model accuracy?
Answer: The calibrated model achieved 86

,question,answer
0,How many rows are in the dataset?,The dataset has 500 rows.
1,What is the average income?,"The mean of income is 85,746.1120."
2,How many missing values are there?,The dataset has no missing values.
3,Which is the best model?,The validation-selected model is Decision Tree...
4,What are the most important features?,"The most influential features are visits, inco..."
5,What is the model accuracy?,"The calibrated model achieved 86.00% accuracy,..."
6,How many clusters were created?,K-Means selected 8 clusters. The silhouette sc...
7,How many anomalies were found?,Isolation Forest identified 25 potential anoma...


In [8]:
# Create Report Charts

target_chart_path = (
    CHART_DIR / "report_target_distribution.png"
)

feature_chart_path = (
    CHART_DIR / "report_feature_importance.png"
)

model_chart_path = (
    CHART_DIR / "report_model_validation.png"
)

cluster_chart_path = (
    CHART_DIR / "report_cluster_purchase_rates.png"
)


plt.figure(figsize=(7, 4))

target_counts = (
    df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
)

target_counts.plot(
    kind="bar",
    color=["#64748B", "#2563EB"],
)

plt.title("Target Distribution")
plt.xlabel("Purchased")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(
    target_chart_path,
    dpi=180,
    bbox_inches="tight",
)
plt.close()


plt.figure(figsize=(8, 4.5))

feature_plot_df = (
    grouped_shap_df.head(10)
    .sort_values("mean_absolute_shap")
)

plt.barh(
    feature_plot_df["original_feature"],
    feature_plot_df["mean_absolute_shap"],
    color="#2563EB",
)

plt.title("Global SHAP Feature Importance")
plt.xlabel("Mean absolute SHAP value")
plt.tight_layout()
plt.savefig(
    feature_chart_path,
    dpi=180,
    bbox_inches="tight",
)
plt.close()


plt.figure(figsize=(9, 5))

validation_plot_df = (
    robust_validation_df.sort_values(
        "cv_f1_macro_mean"
    )
)

plt.barh(
    validation_plot_df["model"],
    validation_plot_df["cv_f1_macro_mean"],
    color="#0F766E",
)

plt.title("Repeated Cross-Validation Macro F1")
plt.xlabel("Macro F1")
plt.tight_layout()
plt.savefig(
    model_chart_path,
    dpi=180,
    bbox_inches="tight",
)
plt.close()


plt.figure(figsize=(9, 5))

plt.bar(
    cluster_profile_df[
        "kmeans_cluster"
    ].astype(str),
    cluster_profile_df[
        "purchase_rate_percentage"
    ],
    color="#7C3AED",
)

plt.title("Purchase Rate by Customer Cluster")
plt.xlabel("Cluster")
plt.ylabel("Purchase rate (%)")
plt.tight_layout()
plt.savefig(
    cluster_chart_path,
    dpi=180,
    bbox_inches="tight",
)
plt.close()

print("Report charts created.")

Report charts created.


In [9]:
# PDF Helper Functions

pdf_styles = getSampleStyleSheet()

pdf_styles.add(
    ParagraphStyle(
        name="ReportTitle",
        parent=pdf_styles["Title"],
        fontSize=24,
        leading=30,
        textColor=colors.HexColor("#172554"),
        alignment=TA_CENTER,
        spaceAfter=20,
    )
)

pdf_styles.add(
    ParagraphStyle(
        name="ReportHeading",
        parent=pdf_styles["Heading2"],
        fontSize=16,
        leading=20,
        textColor=colors.HexColor("#1D4ED8"),
        spaceBefore=10,
        spaceAfter=10,
    )
)

pdf_styles.add(
    ParagraphStyle(
        name="ReportBody",
        parent=pdf_styles["BodyText"],
        fontSize=10,
        leading=15,
        spaceAfter=6,
    )
)


def add_pdf_bullets(story, items):
    for item in items:
        story.append(
            Paragraph(
                f"- {item}",
                pdf_styles["ReportBody"],
            )
        )


def add_pdf_image(
    story,
    image_path,
    width=6.5 * inch,
):
    if not Path(image_path).exists():
        return

    image = PDFImage(str(image_path))

    ratio = image.imageHeight / image.imageWidth

    image.drawWidth = width
    image.drawHeight = width * ratio

    story.append(image)
    story.append(Spacer(1, 12))

In [10]:
# Generate PDF Reports

PDF_PATH = (
    REPORT_DIR / "Intelligent_BI_Copilot_Report.pdf"
)

pdf_document = SimpleDocTemplate(
    str(PDF_PATH),
    pagesize=A4,
    rightMargin=40,
    leftMargin=40,
    topMargin=40,
    bottomMargin=40,
)

pdf_story = []

pdf_story.append(
    Paragraph(
        "Intelligent Business Intelligence Copilot",
        pdf_styles["ReportTitle"],
    )
)

pdf_story.append(
    Paragraph(
        "Automated Statistical, Machine Learning "
        "and Business Insight Report",
        pdf_styles["Heading3"],
    )
)

pdf_story.append(Spacer(1, 20))

pdf_story.append(
    Paragraph(
        "Executive Summary",
        pdf_styles["ReportHeading"],
    )
)

add_pdf_bullets(
    pdf_story,
    business_insights[:6],
)

pdf_story.append(
    Paragraph(
        "Dataset Overview",
        pdf_styles["ReportHeading"],
    )
)

overview_table_data = [
    ["Metric", "Value"],
    ["Rows", f"{dataset_overview['rows']:,}"],
    ["Columns", dataset_overview["columns"]],
    [
        "Missing values",
        dataset_overview["missing_values"],
    ],
    [
        "Duplicate rows",
        dataset_overview["duplicate_rows"],
    ],
    [
        "Purchase rate",
        f"{dataset_overview['purchase_rate'] * 100:.2f}%",
    ],
]

overview_table = Table(
    overview_table_data,
    colWidths=[2.7 * inch, 2.7 * inch],
)

overview_table.setStyle(
    TableStyle([
        (
            "BACKGROUND",
            (0, 0),
            (-1, 0),
            colors.HexColor("#1D4ED8"),
        ),
        (
            "TEXTCOLOR",
            (0, 0),
            (-1, 0),
            colors.white,
        ),
        (
            "GRID",
            (0, 0),
            (-1, -1),
            0.5,
            colors.HexColor("#CBD5E1"),
        ),
        (
            "ROWBACKGROUNDS",
            (0, 1),
            (-1, -1),
            [
                colors.white,
                colors.HexColor("#F8FAFC"),
            ],
        ),
        (
            "PADDING",
            (0, 0),
            (-1, -1),
            7,
        ),
    ])
)

pdf_story.append(overview_table)
pdf_story.append(Spacer(1, 16))

add_pdf_image(
    pdf_story,
    target_chart_path,
)

pdf_story.append(PageBreak())

pdf_story.append(
    Paragraph(
        "Supervised Model Performance",
        pdf_styles["ReportHeading"],
    )
)

model_table_data = [
    ["Metric", "Calibrated result"],
    [
        "Accuracy",
        f"{calibrated_metrics['accuracy']:.4f}",
    ],
    [
        "Balanced accuracy",
        f"{calibrated_metrics['balanced_accuracy']:.4f}",
    ],
    [
        "Macro F1",
        f"{calibrated_metrics['macro_f1']:.4f}",
    ],
    [
        "ROC-AUC",
        f"{calibrated_metrics['roc_auc']:.4f}",
    ],
    [
        "Log loss",
        f"{calibrated_metrics['log_loss']:.4f}",
    ],
    [
        "Brier score",
        f"{calibrated_metrics['brier_score']:.4f}",
    ],
]

model_table = Table(
    model_table_data,
    colWidths=[2.7 * inch, 2.7 * inch],
)

model_table.setStyle(
    TableStyle([
        (
            "BACKGROUND",
            (0, 0),
            (-1, 0),
            colors.HexColor("#0F766E"),
        ),
        (
            "TEXTCOLOR",
            (0, 0),
            (-1, 0),
            colors.white,
        ),
        (
            "GRID",
            (0, 0),
            (-1, -1),
            0.5,
            colors.HexColor("#CBD5E1"),
        ),
        (
            "PADDING",
            (0, 0),
            (-1, -1),
            7,
        ),
    ])
)

pdf_story.append(model_table)
pdf_story.append(Spacer(1, 16))

add_pdf_image(
    pdf_story,
    model_chart_path,
)

pdf_story.append(
    Paragraph(
        "Model Explainability",
        pdf_styles["ReportHeading"],
    )
)

add_pdf_bullets(
    pdf_story,
    [
        (
            f"Top SHAP features: "
            f"{', '.join(top_shap_features)}."
        ),
        (
            "The calibrated model is used for "
            "probability predictions."
        ),
        (
            "The underlying Decision Tree is used "
            "for SHAP and decision-rule explanations."
        ),
    ],
)

add_pdf_image(
    pdf_story,
    feature_chart_path,
)

pdf_story.append(PageBreak())

pdf_story.append(
    Paragraph(
        "Unsupervised Learning",
        pdf_styles["ReportHeading"],
    )
)

add_pdf_bullets(
    pdf_story,
    business_insights[6:],
)

add_pdf_image(
    pdf_story,
    cluster_chart_path,
)

pdf_story.append(
    Paragraph(
        "Recommendations",
        pdf_styles["ReportHeading"],
    )
)

add_pdf_bullets(
    pdf_story,
    recommendations,
)

pdf_story.append(
    Paragraph(
        "Limitations",
        pdf_styles["ReportHeading"],
    )
)

add_pdf_bullets(
    pdf_story,
    limitations,
)

pdf_document.build(pdf_story)

print("PDF created:", PDF_PATH)

PDF created: e:\Practice_PROJECTS\BI_Intelligence\artifacts\reports\Intelligent_BI_Copilot_Report.pdf


In [11]:
# Poerpoint Helper Fuctions

PPTX_PATH = (
    REPORT_DIR
    / "Intelligent_BI_Copilot_Presentation.pptx"
)

presentation = Presentation()

presentation.slide_width = Inches(13.333)
presentation.slide_height = Inches(7.5)

NAVY = RGBColor(23, 37, 84)
BLUE = RGBColor(37, 99, 235)
DARK = RGBColor(30, 41, 59)
WHITE = RGBColor(255, 255, 255)


def style_slide_title(title_shape):
    title_shape.text_frame.paragraphs[
        0
    ].font.name = "Aptos Display"

    title_shape.text_frame.paragraphs[
        0
    ].font.size = Pt(28)

    title_shape.text_frame.paragraphs[
        0
    ].font.color.rgb = NAVY


def add_title_slide(title, subtitle):
    slide = presentation.slides.add_slide(
        presentation.slide_layouts[0]
    )

    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = NAVY

    slide.shapes.title.text = title

    title_paragraph = (
        slide.shapes.title
        .text_frame.paragraphs[0]
    )

    title_paragraph.font.name = "Aptos Display"
    title_paragraph.font.size = Pt(32)
    title_paragraph.font.color.rgb = WHITE

    slide.placeholders[1].text = subtitle

    subtitle_paragraph = (
        slide.placeholders[1]
        .text_frame.paragraphs[0]
    )

    subtitle_paragraph.font.name = "Aptos"
    subtitle_paragraph.font.size = Pt(18)
    subtitle_paragraph.font.color.rgb = WHITE

    return slide


def add_bullet_slide(title, items):
    slide = presentation.slides.add_slide(
        presentation.slide_layouts[1]
    )

    slide.shapes.title.text = title
    style_slide_title(slide.shapes.title)

    text_frame = slide.placeholders[
        1
    ].text_frame

    text_frame.clear()

    for index, item in enumerate(items):
        paragraph = (
            text_frame.paragraphs[0]
            if index == 0
            else text_frame.add_paragraph()
        )

        paragraph.text = str(item)
        paragraph.level = 0
        paragraph.font.name = "Aptos"
        paragraph.font.size = Pt(18)
        paragraph.font.color.rgb = DARK
        paragraph.space_after = Pt(10)

    return slide


def add_chart_slide(title, image_path):
    slide = presentation.slides.add_slide(
        presentation.slide_layouts[5]
    )

    slide.shapes.title.text = title
    style_slide_title(slide.shapes.title)

    slide.shapes.add_picture(
        str(image_path),
        Inches(1.3),
        Inches(1.35),
        width=Inches(10.7),
        height=Inches(5.6),
    )

    return slide

In [12]:
# Generate Poerpoints

add_title_slide(
    "Intelligent Business Intelligence Copilot",
    (
        "Automated data quality, statistics, "
        "machine learning and business insights"
    ),
)

add_bullet_slide(
    "Executive Summary",
    business_insights[:6],
)

add_bullet_slide(
    "Dataset Overview",
    [
        f"Rows: {dataset_overview['rows']:,}",
        f"Columns: {dataset_overview['columns']}",
        (
            f"Purchase rate: "
            f"{dataset_overview['purchase_rate'] * 100:.2f}%"
        ),
        (
            f"Missing values: "
            f"{dataset_overview['missing_values']}"
        ),
        (
            f"Duplicate rows: "
            f"{dataset_overview['duplicate_rows']}"
        ),
    ],
)

add_chart_slide(
    "Target Distribution",
    target_chart_path,
)

add_chart_slide(
    "Repeated Model Validation",
    model_chart_path,
)

add_bullet_slide(
    "Final Model Performance",
    [
        (
            f"Model: "
            f"{final_model_summary['final_model']}"
        ),
        (
            f"Accuracy: "
            f"{calibrated_metrics['accuracy']:.4f}"
        ),
        (
            f"Balanced accuracy: "
            f"{calibrated_metrics['balanced_accuracy']:.4f}"
        ),
        (
            f"Macro F1: "
            f"{calibrated_metrics['macro_f1']:.4f}"
        ),
        (
            f"ROC-AUC: "
            f"{calibrated_metrics['roc_auc']:.4f}"
        ),
        (
            f"Log loss: "
            f"{calibrated_metrics['log_loss']:.4f}"
        ),
    ],
)

add_chart_slide(
    "Feature Importance",
    feature_chart_path,
)

add_chart_slide(
    "Customer Segments",
    cluster_chart_path,
)

add_bullet_slide(
    "Recommendations",
    recommendations,
)

add_bullet_slide(
    "Limitations",
    limitations,
)

presentation.save(PPTX_PATH)

print("PowerPoint created:", PPTX_PATH)

PowerPoint created: e:\Practice_PROJECTS\BI_Intelligence\artifacts\reports\Intelligent_BI_Copilot_Presentation.pptx


In [13]:
# Save insights and Q&A

insight_package = {
    "dataset_overview": dataset_overview,
    "business_insights": business_insights,
    "recommendations": recommendations,
    "limitations": limitations,
    "qa_examples": qa_results,
    "model_metrics": calibrated_metrics,
    "explanation_summary":
        explanation_summary,
    "unsupervised_summary":
        unsupervised_summary,
}

with open(
    REPORT_DIR / "automated_insights.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        insight_package,
        file,
        indent=4,
    )

qa_results_df.to_csv(
    REPORT_DIR / "sample_qa_results.csv",
    index=False,
)

print("Insights and Q&A saved.")

Insights and Q&A saved.


In [14]:
# Final Verification

required_report_files = [
    REPORT_DIR
    / "Intelligent_BI_Copilot_Report.pdf",
    REPORT_DIR
    / "Intelligent_BI_Copilot_Presentation.pptx",
    REPORT_DIR
    / "automated_insights.json",
    REPORT_DIR
    / "sample_qa_results.csv",
]

for path in required_report_files:
    print(
        f"{path.name}: "
        f"{path.exists()} "
        f"({path.stat().st_size if path.exists() else 0:,} bytes)"
    )

Intelligent_BI_Copilot_Report.pdf: True (193,761 bytes)
Intelligent_BI_Copilot_Presentation.pptx: True (143,769 bytes)
automated_insights.json: True (5,508 bytes)
sample_qa_results.csv: True (852 bytes)
